<div dir="rtl">
    
# קבוצה 18: אמיתי מרמור

</div>

In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import re
import numpy as np

<div dir="rtl">
    
## טעינת הטבלה הבסיסית
----------
</div>

In [4]:
path = r"C:\Users\User\Downloads\title.basics.tsv.gz" 
df = pd.read_csv(path, sep='\t')

In [5]:
df

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894,\N,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,\N,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,\N,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892,\N,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893,\N,1,Short
...,...,...,...,...,...,...,...,...,...
12445833,tt9916848,tvEpisode,Episode #3.17,Episode #3.17,0,2009,\N,\N,Drama
12445834,tt9916850,tvEpisode,Episode #3.19,Episode #3.19,0,2010,\N,\N,Drama
12445835,tt9916852,tvEpisode,Episode #3.20,Episode #3.20,0,2010,\N,\N,Drama
12445836,tt9916856,short,The Wind,The Wind,0,2015,\N,27,Short


In [6]:
df = df[df['titleType'] == 'movie'] # הגבלה כדי לקבל רק סרטים
df['startYear'] = pd.to_numeric(df['startYear'], errors='coerce') # המרת שנת יציאת הסרט למספר
df['runtimeMinutes'] = pd.to_numeric(df['runtimeMinutes'], errors='coerce') # המרת זמן הסרט למספר 
df = df[(df['runtimeMinutes'].between(60, 300))] # הגבלת זמן הסרט בין 60 ל-300 דקות
df = df[df['startYear'] <= 2024] # הגבלת שנת יציאת הסרט, עד 2024 כולל

C:\Users\User\AppData\Local\Temp\ipykernel_8060\1688354023.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['startYear'] = pd.to_numeric(df['startYear'], errors='coerce') # המרת שנת יציאת הסרט למספר
C:\Users\User\AppData\Local\Temp\ipykernel_8060\1688354023.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['runtimeMinutes'] = pd.to_numeric(df['runtimeMinutes'], errors='coerce') # המרת זמן הסרט למספר


In [7]:
mask = (df['primaryTitle'].str.slice(0, 2) >= 'Sm') & (df['primaryTitle'].str.slice(0, 2) <= 'Sz') # Sm לבין Sz חיתוך הסרטים שבין האותיות

df_filtered = df[mask].copy()

print(f"Number of movies found between Sm and Sz: {len(df_filtered)}")
print("First few titles:")
print(df_filtered['primaryTitle'].head())

Number of movies found between Sm and Sz: 14082
First few titles:
4556                Soldiers of Fortune
4582                        Stormfågeln
4590    Stuart Webbs: Das Panzergewölbe
6022                          Stingaree
6033                           Strejken
Name: primaryTitle, dtype: object


In [8]:
cols_to_drop = ['titleType', 'originalTitle', 'isAdult', 'endYear']
df_filtered = df_filtered.drop(columns=cols_to_drop) # השארת השורות הרצויות והרדת השורות האחרות
df_filtered

,tconst,primaryTitle,startYear,runtimeMinutes,genres
4556,tt0004613,Soldiers of Fortune,1914.0,60.0,Drama
4582,tt0004639,Stormfågeln,1914.0,64.0,Drama
4590,tt0004647,Stuart Webbs: Das Panzergewölbe,1914.0,60.0,"Action,Crime"
6022,tt0006097,Stingaree,1915.0,250.0,Drama
6033,tt0006108,Strejken,1914.0,78.0,Drama
...,...,...,...,...,...
12437377,tt9898534,Sri Andalas,1966.0,105.0,"Family,Fantasy,Mystery"
12438002,tt9899972,Spring Comes Late,1980.0,81.0,Drama
12442889,tt9910530,Svartklubb,2020.0,79.0,Horror
12444104,tt9913056,Swarm Season,2019.0,86.0,Documentary


-----

<div dir="rtl">
    
## טעינת טבלת הדירוגים
------
</div>

In [11]:
path2 = r"C:\Users\User\Downloads\title.ratings.tsv.gz" 

df2 = pd.read_csv(path2, sep='\t')

In [12]:
df2

,tconst,averageRating,numVotes
0,tt0000001,5.7,2207
1,tt0000002,5.5,314
2,tt0000003,6.4,2318
3,tt0000004,5.1,198
4,tt0000005,6.2,3043
...,...,...,...
1662742,tt9916846,5.3,7
1662743,tt9916848,5.2,7
1662744,tt9916850,6.0,7
1662745,tt9916852,5.7,7


<div dir="rtl">

## איחוד הטבלאות
------
</div>

In [14]:
df_combined = pd.merge(df_filtered, df2, on='tconst', how='inner') # איחוד טבלת הבסיס עם טבלת הרייטינג

In [15]:
df_combined

,tconst,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes
0,tt0004613,Soldiers of Fortune,1914.0,60.0,Drama,6.6,17
1,tt0006108,Strejken,1914.0,78.0,Drama,7.9,16
2,tt0007361,Snow White,1916.0,63.0,"Fantasy,Romance",6.2,513
3,tt0007418,Suzanne,1916.0,83.0,"Drama,Romance",6.2,21
4,tt0008634,Straight Shooting,1917.0,62.0,Western,6.4,855
...,...,...,...,...,...,...,...
10383,tt9893078,Sublime,2019.0,93.0,"Biography,Documentary,Music",8.0,14
10384,tt9897230,Smiling Georgia,2023.0,62.0,Documentary,7.3,61
10385,tt9910530,Svartklubb,2020.0,79.0,Horror,5.7,59
10386,tt9913056,Swarm Season,2019.0,86.0,Documentary,6.7,39


-------

<div dir="rtl">
    
## טעינת טבלת השחקנים
------
</div>

In [18]:
path3 = r"C:\Users\User\Downloads\title.principals.tsv.gz" 

df3 = pd.read_csv(path3, sep='\t')

In [19]:
df3

,tconst,ordering,nconst,category,job,characters
0,tt0000001,1,nm1588970,self,\N,"[""Self""]"
1,tt0000001,2,nm0005690,director,\N,\N
2,tt0000001,3,nm0005690,producer,producer,\N
3,tt0000001,4,nm0374658,cinematographer,director of photography,\N
4,tt0000002,1,nm0721526,director,\N,\N
...,...,...,...,...,...,...
99025886,tt9916880,17,nm0996406,director,principal director,\N
99025887,tt9916880,18,nm1482639,writer,\N,\N
99025888,tt9916880,19,nm2586970,writer,books,\N
99025889,tt9916880,20,nm1594058,producer,producer,\N


In [20]:
df3 = df3[(df3['category'] == 'actor') | (df3['category'] == 'actress')] # השארת שחקנים ושחקניות בלבד
cols_to_drop3 = ['job', 'characters']
df3 = df3.drop(columns=cols_to_drop3) # השארת שורות רצויות והורדת שורות לא רצויות
df3

,tconst,ordering,nconst,category
14,tt0000005,1,nm0443482,actor
15,tt0000005,2,nm0653042,actor
17,tt0000007,1,nm0179163,actor
18,tt0000007,2,nm0183947,actor
24,tt0000008,1,nm0653028,actor
...,...,...,...,...
99025881,tt9916880,12,nm2676923,actress
99025882,tt9916880,13,nm2676923,actress
99025883,tt9916880,14,nm2676923,actress
99025884,tt9916880,15,nm1469295,actress


In [21]:
df3 = df3.sort_values(by=['tconst', 'ordering']) # מיון לפי קוד הסרט ומיון בתוך הסרט לפי סדר השחקנים
df_top5 = df3.groupby('tconst').head(5) # שמירת חמשת השחקנים הראשונים
df_top5

,tconst,ordering,nconst,category
14,tt0000005,1,nm0443482,actor
15,tt0000005,2,nm0653042,actor
17,tt0000007,1,nm0179163,actor
18,tt0000007,2,nm0183947,actor
24,tt0000008,1,nm0653028,actor
...,...,...,...,...
99025870,tt9916880,1,nm2784764,actress
99025871,tt9916880,2,nm1483166,actor
99025872,tt9916880,3,nm1483166,actor
99025873,tt9916880,4,nm1483166,actor


In [22]:
df_actors_lists = df_top5.groupby('tconst')['nconst'].apply(list).reset_index() # קיבוץ חמשת השחקנים הראשונים בסרט לרשימה
df_actors_lists.rename(columns={'nconst': 'lead_actors_ids'}, inplace=True) # שינוי שם העמודה
df_actors_lists = df_actors_lists[df_actors_lists['tconst'].isin(df_combined['tconst'])] # ווידוא שברשימת השחקנים שלנו יישארו רק סרטים שמופיעים בטבלה המאוחדת
df_actors_lists

,tconst,lead_actors_ids
3548,tt0004613,"[nm0267914, nm0820821, nm0695466, nm0823225, n..."
4895,tt0006108,"[nm0803705, nm0803705, nm0526275, nm0414887, n..."
6011,tt0007361,"[nm0191818, nm0354878, nm0103958, nm0913338, n..."
6068,tt0007418,"[nm0334788, nm1057025, nm0874575, nm0993508, n..."
7175,tt0008634,"[nm0002503, nm0497200, nm0077320, nm0540485, n..."
...,...,...
6692254,tt9878650,"[nm9705526, nm9544527, nm10180505, nm8653223, ..."
6692282,tt9878866,"[nm9024097, nm10001111, nm8584663, nm2767828, ..."
6696617,tt9893078,"[nm1454500, nm1455964, nm1453527]"
6701543,tt9910530,"[nm8821891, nm7288448, nm10900172, nm9955536, ..."


--------

<div dir="rtl">
    
## איחוד שלושת הטבלאות
-------
</div>

In [25]:
df_combined = pd.merge(df_combined, df_actors_lists, on='tconst', how='inner') # איחוד הטבלה המאוחדת בטבלת השחקנים
df_combined

,tconst,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,lead_actors_ids
0,tt0004613,Soldiers of Fortune,1914.0,60.0,Drama,6.6,17,"[nm0267914, nm0820821, nm0695466, nm0823225, n..."
1,tt0006108,Strejken,1914.0,78.0,Drama,7.9,16,"[nm0803705, nm0803705, nm0526275, nm0414887, n..."
2,tt0007361,Snow White,1916.0,63.0,"Fantasy,Romance",6.2,513,"[nm0191818, nm0354878, nm0103958, nm0913338, n..."
3,tt0007418,Suzanne,1916.0,83.0,"Drama,Romance",6.2,21,"[nm0334788, nm1057025, nm0874575, nm0993508, n..."
4,tt0008634,Straight Shooting,1917.0,62.0,Western,6.4,855,"[nm0002503, nm0497200, nm0077320, nm0540485, n..."
...,...,...,...,...,...,...,...,...
9314,tt9878650,Surviving the Storm,2023.0,113.0,Drama,4.4,36,"[nm9705526, nm9544527, nm10180505, nm8653223, ..."
9315,tt9878866,Subha Love,2019.0,128.0,Romance,5.3,12,"[nm9024097, nm10001111, nm8584663, nm2767828, ..."
9316,tt9893078,Sublime,2019.0,93.0,"Biography,Documentary,Music",8.0,14,"[nm1454500, nm1455964, nm1453527]"
9317,tt9910530,Svartklubb,2020.0,79.0,Horror,5.7,59,"[nm8821891, nm7288448, nm10900172, nm9955536, ..."


<div dir="rtl">
    
### בדיקת איכות הנתונים מהטבלאות

</div>

In [27]:
total_checked = len(df_combined) # כמות הסרטים (מספר השורות)

tconst_missing = df_combined['tconst'].isna().sum() # סכימת הערכים החסרים בעמודה
primaryTitle_missing = df_combined['primaryTitle'].isna().sum() # סכימת הערכים החסרים בעמודה
startYear_missing = df_combined['startYear'].isna().sum() # סכימת הערכים החסרים בעמודה
runtimeMinutes_missing = df_combined['runtimeMinutes'].isna().sum() # סכימת הערכים החסרים בעמודה
genres_missing = df_combined['genres'].isna().sum() # סכימת הערכים החסרים בעמודה
averageRating_missing = df_combined['averageRating'].isna().sum() # סכימת הערכים החסרים בעמודה
numVotes_missing = df_combined['numVotes'].isna().sum() # סכימת הערכים החסרים בעמודה
lead_actors_ids_missing = df_combined['lead_actors_ids'].isna().sum() # סכימת הערכים החסרים בעמודה

print(f"--- סיכום איכות נתונים (מתוך {total_checked} סרטים) ---")
print(f"מזהה (tconst) חסר:      {tconst_missing} ({((tconst_missing/total_checked)*100):.1f}%)") # הצגת המספרים גם באחוזים
print(f"כותרת חסרה:            {primaryTitle_missing} ({((primaryTitle_missing/total_checked)*100):.1f}%)") # הצגת המספרים גם באחוזים
print(f"שנת יציאה חסרה:        {startYear_missing} ({((startYear_missing/total_checked)*100):.1f}%)") # הצגת המספרים גם באחוזים
print(f"זמן הרצה חסר:          {runtimeMinutes_missing} ({((runtimeMinutes_missing/total_checked)*100):.1f}%)") # הצגת המספרים גם באחוזים
print(f"ז'אנרים חסרים:         {genres_missing} ({((genres_missing/total_checked)*100):.1f}%)") # הצגת המספרים גם באחוזים
print(f"דירוג ממוצע חסר:       {averageRating_missing} ({((averageRating_missing/total_checked)*100):.1f}%)") # הצגת המספרים גם באחוזים
print(f"מספר קולות חסר:        {numVotes_missing} ({((numVotes_missing/total_checked)*100):.1f}%)") # הצגת המספרים גם באחוזים
print(f"מזהי שחקנים חסרים:     {lead_actors_ids_missing} ({((lead_actors_ids_missing/total_checked)*100):.1f}%)") # הצגת המספרים גם באחוזים

--- סיכום איכות נתונים (מתוך 9319 סרטים) ---
מזהה (tconst) חסר:      0 (0.0%)
כותרת חסרה:            0 (0.0%)
שנת יציאה חסרה:        0 (0.0%)
זמן הרצה חסר:          0 (0.0%)
ז'אנרים חסרים:         0 (0.0%)
דירוג ממוצע חסר:       0 (0.0%)
מספר קולות חסר:        0 (0.0%)
מזהי שחקנים חסרים:     0 (0.0%)


<div dir="rtl">
    
#### ניכר כי החיבור שנעשה בוצע בעזרת חיבור ללא חורים על מנת לקבל מידע כמה שיותר מקיף על הסרטים

</div>

_________

<div dir="rtl">
    
## יצירת טבלת 5,000 הסרטים המובילים
-------
</div>

In [31]:
df_popular = df_combined.sort_values(by='numVotes', ascending=False).head(5000).copy() # חיתוך ל-5,000 סרטים
df_popular

,tconst,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,lead_actors_ids
1423,tt0076759,Star Wars: Episode IV - A New Hope,1977.0,121.0,"Action,Adventure,Fantasy",8.6,1566475,"[nm0000434, nm0000148, nm0000402, nm0000027, n..."
1508,tt0080684,Star Wars: Episode V - The Empire Strikes Back,1980.0,124.0,"Action,Adventure,Fantasy",8.7,1501332,"[nm0000434, nm0000148, nm0000402, nm0001850, n..."
1611,tt0086190,Star Wars: Episode VI - Return of the Jedi,1983.0,131.0,"Action,Adventure,Fantasy",8.3,1206896,"[nm0000434, nm0000148, nm0000402, nm0001850, n..."
6989,tt2488496,Star Wars: Episode VII - The Force Awakens,2015.0,138.0,"Action,Adventure,Sci-Fi",7.7,1024047,"[nm5397459, nm3915784, nm1209966, nm1727304, n..."
4881,tt10872600,Spider-Man: No Way Home,2021.0,148.0,"Action,Adventure,Fantasy",8.1,1015964,"[nm4043618, nm4043618, nm3918035, nm1212722, n..."
...,...,...,...,...,...,...,...,...
5046,tt11663918,Sweethurt,2020.0,93.0,Comedy,6.6,96,"[nm11306490, nm11306491, nm9958894, nm11306492..."
7151,tt27164431,Space Frank,2024.0,98.0,"Comedy,Fantasy,Sci-Fi",4.6,96,"[nm2539875, nm2144988, nm0486619, nm0694907, n..."
1276,tt0070307,"Story of Karate, Fists, and Beans",1973.0,92.0,"Action,Adventure,Comedy",4.5,96,"[nm0715381, nm0400084, nm0948973, nm0562841, n..."
5921,tt15663006,Soyrik,2022.0,109.0,\N,8.7,96,"[nm13227784, nm9717490, nm6746230, nm13227783,..."


________

<div dir="rtl">
    
## בדיקת 50 סרטים אחרונים
-------
</div>

In [34]:
def get_wiki_movie_data(title, year): # פונקציה שמקבלת שם סרט ושנה ומחזירה נתונים מויקיפדיה
    headers = {'User-Agent': 'MovieDataProject/2.5 (contact: yourname@email.com)'}
    search_url = "https://en.wikipedia.org/w/api.php"
    params = {"action": "query", "list": "search", "format": "json", "srsearch": f"{title} {int(year)} film"} # פרמטרים לחיפוש: שם הסרט + שנה + המילה
    
    try:
        s_res = requests.get(search_url, params=params, headers=headers, timeout=5).json() # שליחת בקשת חיפוש לויקיפדיה וקבלת תוצאות JSON
        if not s_res.get('query', {}).get('search'): return None
        best_title = s_res['query']['search'][0]['title'] # לוקח את הכותרת הכי רלוונטית
        
        url = f"https://en.wikipedia.org/wiki/{best_title.replace(' ', '_')}"
        res = requests.get(url, headers=headers, timeout=5)
        if res.status_code != 200: return None
        
        soup = BeautifulSoup(res.text, 'html.parser')
        infobox = soup.find("table", class_="infobox") # מציאת הטבלה
        
        movie_data = {"Language": None, "Country": None, "Budget": None, "BoxOffice": None, "Plot": None} # אתחול המילון שיכיל את הנתונים שנאסוף
        
        if infobox:
            for row in infobox.find_all("tr"):
                th, td = row.find("th"), row.find("td")  # כותרת השורה והערך שלה
                if th and td:
                    lbl = th.text.strip().lower()
                    val = re.sub(r'\[\s*\d+\s*\]', '', td.get_text(separator=" ").strip()).replace('\xa0', ' ').replace('\n', ' ')
                    # שיוך הערך לעמודה המתאימה לפי שם השדה
                    if "language" in lbl: movie_data["Language"] = val
                    elif "country" in lbl or "countries" in lbl: movie_data["Country"] = val
                    elif "budget" in lbl: movie_data["Budget"] = val
                    elif "box office" in lbl or "gross" in lbl: movie_data["BoxOffice"] = val



        for h_id in ['Plot', 'Synopsis', 'Summary', 'Story', 'Premise']: # חיפוש קטע העלילה תחת כותרות שונות
            plot_h = soup.find('h2', id=h_id) or soup.find('span', id=h_id)
            if plot_h:
                container = plot_h.find_parent('div', class_='mw-heading') or plot_h # מציאת הקונטיינר של הכותרת
                for sib in container.find_next_siblings(): # מעבר על האלמנטים שאחרי הכותרת
                    if sib.name == 'p' and len(sib.text.strip()) > 30:
                        movie_data["Plot"] = sib.text.strip()[:400] + "..."
                        break
                    if sib.name in ['h2', 'h3']: break
        return movie_data
    except: return None


df_test_last = df_popular.tail(50).copy()

print(f"🚀 מריץ בדיקה על 50 הסרטים האחרונים בטבלה (מקומות 4951-5000)...")

for idx, row in df_test_last.iterrows():
    data = get_wiki_movie_data(row['primaryTitle'], row['startYear'])
    if data:
        for col, val in data.items():
            df_test_last.at[idx, col] = val
        status = "✅" if data['Plot'] else "⚠️ (ללא עלילה)"
    else:
        status = "❌"
    
    print(f"סרט: {row['primaryTitle'][:25]:<25} | {status}")
    time.sleep(1.2)

print("\n--- ✨ הבדיקה הסתיימה! ---")
df_test_last[['primaryTitle', 'startYear', 'Language', 'Country', 'Plot']].head(10)

🚀 מריץ בדיקה על 50 הסרטים האחרונים בטבלה (מקומות 4951-5000)...
סרט: Stormy Weather            | ✅
סרט: Soul and Flesh            | ⚠️ (ללא עלילה)
סרט: Smalltown Boys            | ⚠️ (ללא עלילה)
סרט: Sparkle and Charm         | ⚠️ (ללא עלילה)
סרט: Streamer                  | ⚠️ (ללא עלילה)
סרט: Swedish Erotica Featurett | ⚠️ (ללא עלילה)
סרט: Son of a General III      | ⚠️ (ללא עלילה)
סרט: Sunstroke at the Beach Re | ⚠️ (ללא עלילה)
סרט: Spin the Bottle           | ✅
סרט: Something Necessary       | ✅
סרט: Superior                  | ✅
סרט: Snow and Salome           | ⚠️ (ללא עלילה)
סרט: Sokaktan Gelen Kadin      | ⚠️ (ללא עלילה)
סרט: Soul Pursuit              | ✅
סרט: Super Mix                 | ✅
סרט: Son-nim-1 Cheo-beon-jjae  | ❌
סרט: Steigler and Steigler     | ✅
סרט: Super                     | ✅
סרט: Striptease                | ⚠️ (ללא עלילה)
סרט: Spanking at School        | ⚠️ (ללא עלילה)
סרט: South of Sanity           | ✅
סרט: Surdina                   | ⚠️ (ללא עלילה)
סרט: Strang

,primaryTitle,startYear,Language,Country,Plot
261,Stormy Weather,1935.0,English,United Kingdom,Sir Duncan Craggs retires from the Colonial Se...
3227,Soul and Flesh,1974.0,None,None,None
6593,Smalltown Boys,2022.0,None,None,None
2656,Sparkle and Charm,1997.0,None,None,None
8667,Streamer,2016.0,None,None,None
2565,Swedish Erotica Featurettes 1,1989.0,None,None,None
2567,Son of a General III,1991.0,Korean Japanese Mandarin,South Korea,None
2527,Sunstroke at the Beach Resort,1973.0,Danish,Denmark,None
3272,Spin the Bottle,1999.0,English,United States,Childhood friends meet up for a reunion....
6921,Something Necessary,2013.0,Swahili,Kenya,Kenya 2007: Following the results of the dispu...


<div dir="rtl">
    
### מטרת בדיקה זו היא לראות את מצב הנתונים. במידה והנתונים בחמישים הסרטים האחרונים מספיק טובים הם גם יהיו טובים בהמשך. הטבלה סודרה כך שהנתונים הכי פחות טובים יהיו בסוף.

</div>

In [36]:
df_test_last.head(50)

,tconst,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,lead_actors_ids,Language,Country,Budget,BoxOffice,Plot
261,tt0028312,Stormy Weather,1935.0,74.0,Comedy,6.1,99,"[nm0909398, nm0528785, nm0036084, nm0362823, n...",English,United Kingdom,None,None,Sir Duncan Craggs retires from the Colonial Se...
3227,tt0236641,Soul and Flesh,1974.0,100.0,"Drama,Romance",6.2,99,"[nm0126381, nm0126381, nm0875152, nm0875152, n...",None,None,None,None,None
6593,tt21276180,Smalltown Boys,2022.0,82.0,Drama,6.2,99,"[nm6754272, nm0719570, nm7289544, nm13866829, ...",None,None,None,None,None
2656,tt0163274,Sparkle and Charm,1997.0,90.0,"Comedy,Drama",7.8,98,"[nm0290280, nm0887170, nm0397212, nm0238004, n...",None,None,None,None,None
8667,tt6380398,Streamer,2016.0,78.0,Drama,5.6,98,"[nm3967526, nm3060119, nm3060119, nm8678121]",None,None,None,None,None
2565,tt0149216,Swedish Erotica Featurettes 1,1989.0,80.0,Adult,7.1,98,"[nm0014959, nm0446178, nm0598570, nm0000561, n...",None,None,None,None,None
2567,tt0150090,Son of a General III,1991.0,110.0,"Action,Crime,Drama",5.4,98,"[nm0661929, nm1059110, nm1067169, nm0793790, n...",Korean Japanese Mandarin,South Korea,None,None,None
2527,tt0145477,Sunstroke at the Beach Resort,1973.0,81.0,Comedy,4.4,98,"[nm0683397, nm0197376, nm0197376, nm0656458, n...",Danish,Denmark,None,None,None
3272,tt0245470,Spin the Bottle,1999.0,97.0,Comedy,5.4,98,"[nm0505177, nm0944853, nm0614337, nm0034006, n...",English,United States,None,None,Childhood friends meet up for a reunion....
6921,tt2400272,Something Necessary,2013.0,85.0,Drama,6.9,98,"[nm5276844, nm5277621, nm5470343, nm5470302, n...",Swahili,Kenya,None,None,Kenya 2007: Following the results of the dispu...


<div dir="rtl">
    
### התוצאות לא היו מספיק ברורות ולכן החלטתי להריץ את הנתונים על 200 השורות האחרונות ולא הסתפקתי רק בחמישים האחרונות

</div>

In [38]:
df_test_last = df_popular.tail(200).copy() # חיתוך הטבלה למאתיים השורות האחרונות

print(f"🚀 מריץ בדיקה על 200 הסרטים האחרונים בטבלה (מקומות 4801-5000)...")

for idx, row in df_test_last.iterrows():
    data = get_wiki_movie_data(row['primaryTitle'], row['startYear'])
    if data: # בדיקה האם נמצא הדף בוויקיפדיה
        for col, val in data.items(): # אם נמצא הדף עוברים על המילון
            df_test_last.at[idx, col] = val # הוספת הערכים שנמצאו
        status = "✅" if data['Plot'] else "⚠️ (ללא עלילה)" # אם נמצאה העלילה תעשה וי ואם לא תעשה אזהרה
    else:
        status = "❌" # אם לא נמצא הדף תעשה איקס
    
    print(f"סרט: {row['primaryTitle'][:25]:<25} | {status}")
    time.sleep(1.2)

print("\n--- ✨ הבדיקה הסתיימה! ---")
df_test_last[['primaryTitle', 'startYear', 'Language', 'Country', 'Plot']].head(10)

🚀 מריץ בדיקה על 200 הסרטים האחרונים בטבלה (מקומות 4801-5000)...
סרט: Surviving the Rush        | ✅
סרט: Stateless                 | ⚠️ (ללא עלילה)
סרט: Songs for a Sloth         | ✅
סרט: Strategia per una mission | ⚠️ (ללא עלילה)
סרט: Swargam Narakam           | ✅
סרט: Stone Bridge              | ✅
סרט: Song of Old Wyoming       | ⚠️ (ללא עלילה)
סרט: Swamp Country             | ✅
סרט: Suburbs                   | ✅
סרט: Soul                      | ✅
סרט: Spook                     | ✅
סרט: Strange But True          | ✅
סרט: Sonata                    | ✅
סרט: Stuck: Intrappolati nell' | ❌
סרט: Sweden Dying to Be Multic | ⚠️ (ללא עלילה)
סרט: Stalker                   | ✅
סרט: Syncopation               | ✅
סרט: Sunday Morning in Victori | ⚠️ (ללא עלילה)
סרט: Sprawiedliwy              | ⚠️ (ללא עלילה)
סרט: Sum of Existence          | ✅
סרט: Sweethearts of the U.S.A. | ⚠️ (ללא עלילה)
סרט: Spartacus                 | ⚠️ (ללא עלילה)
סרט: Stigmata                  | ✅
סרט: Spookiz: The Movie    

,primaryTitle,startYear,Language,Country,Plot
4439,Surviving the Rush,2007.0,English,United States,"Three years after the events of Rush Hour 2,[n..."
8588,Stateless,2020.0,None,None,None
4848,Songs for a Sloth,2021.0,English,United States,A herd of prehistoric animals is migrating sou...
3106,Strategia per una missione di morte,1979.0,None,None,None
3389,Swargam Narakam,1975.0,Telugu,India,The plot revolves around three couples. The fi...
2180,Stone Bridge,1996.0,English,United States,"On an Oklahoma farm in 1969, young Jo Thornton..."
522,Song of Old Wyoming,1945.0,English,United States,None
2642,Swamp Country,1966.0,English,United States,"Janeen, a young girl living in Georgia's Okefe..."
4132,Suburbs,2004.0,English,United States,"Brittany Aarons is a regular teenage girl, one..."
4206,Soul,2003.0,English,United States,"In the year 2003, 13-year-old Bethany Hamilton..."


In [39]:
total_checked = len(df_test_last) # כמות הסרטים שנבדקו

def count_missing(column_name):
    return df_test_last[column_name].apply(lambda x: x is None or x == 'None' or str(x) == 'nan').sum() # בדיקה לכמות הערכים הטובים בטבלה

plot_missing = count_missing('Plot')
budget_missing = count_missing('Budget')
lang_missing = count_missing('Language')
country_missing = count_missing('Country')
box_office_missing = count_missing('BoxOffice')

print(f"--- סיכום איכות נתונים (מתוך {total_checked} סרטים) ---")
print(f"עלילה חסרה:    {plot_missing} ({((plot_missing/total_checked)*100):.1f}%)")
print(f"תקציב חסר:    {budget_missing} ({((budget_missing/total_checked)*100):.1f}%)")
print(f"שפה חסרה:     {lang_missing} ({((lang_missing/total_checked)*100):.1f}%)")
print(f"מדינה חסרה:    {country_missing} ({((country_missing/total_checked)*100):.1f}%)")
print(f"הכנסות חסרות:  {box_office_missing} ({((box_office_missing/total_checked)*100):.1f}%)")

--- סיכום איכות נתונים (מתוך 200 סרטים) ---
עלילה חסרה:    108 (54.0%)
תקציב חסר:    153 (76.5%)
שפה חסרה:     88 (44.0%)
מדינה חסרה:    88 (44.0%)
הכנסות חסרות:  151 (75.5%)


<div dir="rtl">
    
### מאחר ומדובר במאתיים הסרטים האחרונים החלטתי להריץ על כל הסרטים כי הנתונים הבאים יותר טובים מאלה שנבדקו

</div>

_________

<div dir="rtl">
    
## הרצת הטבלה המלאה
-------
</div>

In [43]:
df_wiki_final = df_popular.copy()
for col in ["Language", "Country", "Budget", "BoxOffice", "Plot"]:
    df_wiki_final[col] = None

start_time = time.time() # שמירת הזמן הנוכחי
print(f"🚀 מתחיל איסוף נתונים ל-5,000 סרטים...")

for i, (idx, row) in enumerate(df_wiki_final.iterrows(), 1):
    data = get_wiki_movie_data(row['primaryTitle'], row['startYear'])
    if data:
        for col, val in data.items():
            df_wiki_final.at[idx, col] = val
            
    if i % 100 == 0:
        elapsed = (time.time() - start_time) / 60 # חישוב זמן הריצה
        df_wiki_final.to_csv("movies_5000_progress.csv", index=False) # גיבוי
        print(f"✅ התקדמות: {i}/5000... ({elapsed:.1f} דקות)") # שליחת הודעה כל 100 סרטים
    
    time.sleep(1.2)

df_wiki_final.to_csv("FINAL_WIKI_DATA_5000.csv", index=False)
print(f"\n✨ סיימנו! הקובץ FINAL_WIKI_DATA_5000.csv מוכן.")

🚀 מתחיל איסוף נתונים ל-5,000 סרטים...
✅ התקדמות: 100/5000... (4.6 דקות)
✅ התקדמות: 200/5000... (8.7 דקות)
✅ התקדמות: 300/5000... (12.7 דקות)
✅ התקדמות: 400/5000... (16.9 דקות)
✅ התקדמות: 500/5000... (20.9 דקות)
✅ התקדמות: 600/5000... (24.9 דקות)
✅ התקדמות: 700/5000... (29.0 דקות)
✅ התקדמות: 800/5000... (33.1 דקות)
✅ התקדמות: 900/5000... (37.1 דקות)
✅ התקדמות: 1000/5000... (41.1 דקות)
✅ התקדמות: 1100/5000... (45.2 דקות)
✅ התקדמות: 1200/5000... (49.2 דקות)
✅ התקדמות: 1300/5000... (53.3 דקות)
✅ התקדמות: 1400/5000... (57.4 דקות)
✅ התקדמות: 1500/5000... (61.6 דקות)
✅ התקדמות: 1600/5000... (65.7 דקות)
✅ התקדמות: 1700/5000... (69.7 דקות)
✅ התקדמות: 1800/5000... (73.8 דקות)
✅ התקדמות: 1900/5000... (77.9 דקות)
✅ התקדמות: 2000/5000... (82.1 דקות)
✅ התקדמות: 2100/5000... (86.3 דקות)
✅ התקדמות: 2200/5000... (90.5 דקות)
✅ התקדמות: 2300/5000... (94.7 דקות)
✅ התקדמות: 2400/5000... (98.9 דקות)
✅ התקדמות: 2500/5000... (103.1 דקות)
✅ התקדמות: 2600/5000... (107.2 דקות)
✅ התקדמות: 2700/5000... (111.6 דקות

In [44]:
df_wiki_final

,tconst,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,lead_actors_ids,Language,Country,Budget,BoxOffice,Plot
1423,tt0076759,Star Wars: Episode IV - A New Hope,1977.0,121.0,"Action,Adventure,Fantasy",8.6,1566475,"[nm0000434, nm0000148, nm0000402, nm0000027, n...",English,United States,$11 million,$775.4 million,"In a period of galactic civil war, Rebel Allia..."
1508,tt0080684,Star Wars: Episode V - The Empire Strikes Back,1980.0,124.0,"Action,Adventure,Fantasy",8.7,1501332,"[nm0000434, nm0000148, nm0000402, nm0001850, n...",English,United States,$30.5 million,$549–$550 million [ i ],Three years after the destruction of the Death...
1611,tt0086190,Star Wars: Episode VI - Return of the Jedi,1983.0,131.0,"Action,Adventure,Fantasy",8.3,1206896,"[nm0000434, nm0000148, nm0000402, nm0001850, n...",English,United States,$32.5–42.7 million,$482 million,One year after Han Solo's capture and imprison...
6989,tt2488496,Star Wars: Episode VII - The Force Awakens,2015.0,138.0,"Action,Adventure,Sci-Fi",7.7,1024047,"[nm5397459, nm3915784, nm1209966, nm1727304, n...",English,United States,$638.9 million (gross) $535.5 million (net),$2.071 billion,"Thirty years after the Battle of Endor,[a] the..."
4881,tt10872600,Spider-Man: No Way Home,2021.0,148.0,"Action,Adventure,Fantasy",8.1,1015964,"[nm4043618, nm4043618, nm3918035, nm1212722, n...",English,United States,$200 million,$1.921 billion,After Quentin Beck frames Peter Parker for his...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5046,tt11663918,Sweethurt,2020.0,93.0,Comedy,6.6,96,"[nm11306490, nm11306491, nm9958894, nm11306492...",None,None,None,None,None
7151,tt27164431,Space Frank,2024.0,98.0,"Comedy,Fantasy,Sci-Fi",4.6,96,"[nm2539875, nm2144988, nm0486619, nm0694907, n...",English,United Kingdom United States,$10.5 million,$148.5 million,"In a prehistoric veld, a tribe of hominins is ..."
1276,tt0070307,"Story of Karate, Fists, and Beans",1973.0,92.0,"Action,Adventure,Comedy",4.5,96,"[nm0715381, nm0400084, nm0948973, nm0562841, n...",None,None,None,None,None
5921,tt15663006,Soyrik,2022.0,109.0,\N,8.7,96,"[nm13227784, nm9717490, nm6746230, nm13227783,...",Marathi,India,None,None,The film was released on 11 February 2022 in t...


<div dir="rtl">
    
### בדיקת הנתונים הסופיים, כמה קיים מכל עמודה

</div>

In [46]:
total_checked = len(df_wiki_final) # כמות סרטים בטבלה

def count_missing(column_name):
    return df_wiki_final[column_name].apply(lambda x: x is None or x == 'None' or str(x) == 'nan').sum() # בדיקה לכמות הערכים הטובים בטבלה 

plot_missing = count_missing('Plot')
budget_missing = count_missing('Budget')
lang_missing = count_missing('Language')
country_missing = count_missing('Country')
box_office_missing = count_missing('BoxOffice')

print(f"--- סיכום איכות נתונים (מתוך {total_checked} סרטים) ---")
print(f"עלילה חסרה:    {plot_missing} ({((plot_missing/total_checked)*100):.1f}%)")
print(f"תקציב חסר:    {budget_missing} ({((budget_missing/total_checked)*100):.1f}%)")
print(f"שפה חסרה:     {lang_missing} ({((lang_missing/total_checked)*100):.1f}%)")
print(f"מדינה חסרה:    {country_missing} ({((country_missing/total_checked)*100):.1f}%)")
print(f"הכנסות חסרות:  {box_office_missing} ({((box_office_missing/total_checked)*100):.1f}%)")

--- סיכום איכות נתונים (מתוך 5000 סרטים) ---
עלילה חסרה:    1743 (34.9%)
תקציב חסר:    3570 (71.4%)
שפה חסרה:     1331 (26.6%)
מדינה חסרה:    1337 (26.7%)
הכנסות חסרות:  3324 (66.5%)


<div dir="rtl">
    
### נאבדו לא מעט נתונים וייתכן כי מדובר בסרטים שלא מופיעים בוויקפדיה, נעשה ניסיון נוסף לאיסוף סרטים

</div> 

________

<div dir="rtl">
    
## ניסיון שני:
-------
</div>

In [50]:
def rescue_missing_movies_to_new_file(df_input, output_filename="rescued_only.csv"): # פונקציה שמנסה להשלים נתונים חסרים לסרטים שאין להם עלילה
    headers = {'User-Agent': 'MovieResearcher/2.0 (Rescue Mission)'}
    
    missing_mask = df_input['Plot'].isna() | (df_input['Plot'] == 'None') | (df_input['Plot'].astype(str) == 'nan')  # מציאת כל הסרטים שחסרה להם עלילה (None, 'None', או 'nan')
    df_missing = df_input[missing_mask].copy()
    
    if len(df_missing) == 0:     # אם אין סרטים חסרים - מסיימים
        print("✅ הכל מלא! לא נמצאו סרטים ללא עלילה ב-df_wiki_final.")
        return None

    print(f"🔍 נמצאו {len(df_missing)} סרטים ללא עלילה. מתחיל סבב הצלה...")
    
    rescued_list = []     # רשימה שתצבור את הסרטים שהצלחנו להשלים
    start_time = time.time()
    
    for i, (idx, row) in enumerate(df_missing.iterrows(), 1): # מעבר על כל סרט חסר
        title = row['primaryTitle']
        year = row['startYear']
        
        search_url = "https://en.wikipedia.org/w/api.php"
        params = {"action": "query", "list": "search", "format": "json", "srsearch": f"{title} {int(year)} film"}
        
        try:
            s_res = requests.get(search_url, params=params, headers=headers, timeout=5).json()
            if s_res.get('query', {}).get('search'): # לוקח את הכותרת הכי רלוונטית מהחיפוש
                correct_title = s_res['query']['search'][0]['title']
                
                new_data = get_wiki_movie_data(correct_title, year)
                
                if new_data: # משלב את הנתונים המקוריים עם הנתונים החדשים
                    new_entry = row.to_dict()
                    new_entry.update(new_data)
                    rescued_list.append(new_entry)
        except:
            continue
            
        if i % 50 == 0:
            elapsed = (time.time() - start_time) / 60
            print(f"🔄 חולצו {len(rescued_list)} סרטים מתוך {i} שנבדקו... ({elapsed:.1f} דקות)")
            if rescued_list:
                pd.DataFrame(rescued_list).to_csv(output_filename, index=False) # שמירת גיבוי
            
        time.sleep(1.2)

    df_rescued = pd.DataFrame(rescued_list)
    df_rescued.to_csv(output_filename, index=False)
    print(f"\n✨ סיימנו! {len(df_rescued)} סרטים חולצו ונשמרו ב- {output_filename}")
    return df_rescued

df_rescued_results = rescue_missing_movies_to_new_file(df_wiki_final)

🔍 נמצאו 1743 סרטים ללא עלילה. מתחיל סבב הצלה...
🔄 חולצו 36 סרטים מתוך 250 שנבדקו... (2.8 דקות)
🔄 חולצו 37 סרטים מתוך 350 שנבדקו... (3.4 דקות)
🔄 חולצו 133 סרטים מתוך 1500 שנבדקו... (13.3 דקות)

✨ סיימנו! 161 סרטים חולצו ונשמרו ב- rescued_only.csv


<div dir="rtl">
    
### מטרת ניסיון זה היא לקחת את הסרטים שחסרה להם העלילה ולעשות להם חילוץ נוסף.
</div>

In [52]:
df_rescued_results.to_csv("df_rescued_results.csv", index=False)
df_rescued_results

,tconst,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,lead_actors_ids,Language,Country,Budget,BoxOffice,Plot
0,tt1185834,Star Wars: The Clone Wars,2008.0,98.0,"Action,Adventure,Animation",6.0,81157,"[nm1782667, nm0296546, nm0296546, nm0437454, n...",English,United States,$8.5 million,$68.5 million,"Early in the Clone Wars,[b] Galactic Republic ..."
1,tt0076729,Smokey and the Bandit,1977.0,96.0,"Action,Adventure,Comedy",7.0,62409,"[nm0000608, nm0000398, nm0715274, nm0377947, n...",English,United States,$4.3 million,$127 million,"Wealthy Texan Big Enos Burdette and his son, L..."
2,tt6038600,Smolensk,2016.0,120.0,"Drama,Thriller",1.2,40448,"[nm1969331, nm2994478, nm0835406, nm0521485, n...",None,None,None,None,None
3,tt12838766,Space Sweepers,2021.0,136.0,"Action,Adventure,Drama",6.5,31068,"[nm3609366, nm3892241, nm5887272, nm1041999, n...",Korean English Spanish Danish French German Ru...,South Korea,₩24 billion (~US$21.2 million),None,None
4,tt0167427,Superstar,1999.0,81.0,"Comedy,Romance",5.3,21591,"[nm0788340, nm0002071, nm0002071, nm0005006, n...",English,United States,$14 million,$30.6 million,"As a child, Mary Katherine Gallagher rescues a..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,tt0128657,"Surf, Sand and Sex",1987.0,76.0,Adult,5.6,98,"[nm0201110, nm0001000, nm0662902, nm0233013, n...",None,None,None,None,None
157,tt0195861,Smoking Cuban Style,1999.0,88.0,"Comedy,Drama",5.8,98,"[nm0004623, nm0951658, nm0018774, nm0039169, n...",None,None,None,None,None
158,tt0155213,Stuart Bliss,1998.0,88.0,"Action,Drama,Thriller",6.4,97,"[nm0954677, nm0492715, nm0492715, nm0569787, n...",None,None,None,None,None
159,tt0180217,Sumnjivo lice,1954.0,85.0,Comedy,7.8,97,"[nm0664608, nm0548589, nm0866658, nm0897283, n...",None,None,None,None,None


<div dir="rtl">
    
### תוצאות הניסיון השני:

</div>

In [54]:
total_checked = len(df_rescued_results) # כמות סרטים בטבלה

def count_missing(column_name):
    return df_rescued_results[column_name].apply(lambda x: x is None or x == 'None' or str(x) == 'nan').sum() # בדיקה לכמות הערכים הטובים בטבלה 

plot_missing = count_missing('Plot')
budget_missing = count_missing('Budget')
lang_missing = count_missing('Language')
country_missing = count_missing('Country')
box_office_missing = count_missing('BoxOffice')

print(f"--- סיכום איכות נתונים (מתוך {total_checked} סרטים) ---")
print(f"עלילה חסרה:    {plot_missing} ({((plot_missing/total_checked)*100):.1f}%)")
print(f"תקציב חסר:    {budget_missing} ({((budget_missing/total_checked)*100):.1f}%)")
print(f"שפה חסרה:     {lang_missing} ({((lang_missing/total_checked)*100):.1f}%)")
print(f"מדינה חסרה:    {country_missing} ({((country_missing/total_checked)*100):.1f}%)")
print(f"הכנסות חסרות:  {box_office_missing} ({((box_office_missing/total_checked)*100):.1f}%)")

--- סיכום איכות נתונים (מתוך 161 סרטים) ---
עלילה חסרה:    141 (87.6%)
תקציב חסר:    141 (87.6%)
שפה חסרה:     100 (62.1%)
מדינה חסרה:    101 (62.7%)
הכנסות חסרות:  141 (87.6%)


<div dir="rtl">
    
### אמנם לפי האחוזים לא חל שיפור, אך מדובר בסרטים נוספים שלא עברו בהצלחה את הניסיון הראשון אך כן את השני!
### נוכל לצרף אותם לסרטים שקיימים ברישמה ובכך לשפר את התוצאות.
</div>

<div dir="rtl">
    
## אך לפני כן:

</div>

_________

<div dir="rtl">
    
## ניסיון שלישי:
-------
</div>

In [59]:
def fetch_movie_metadata_v3(movie_title, movie_year):
    entry_data = {"Language": None, "Country": None, "Budget": None, "BoxOffice": None, "Plot": None}
    web_headers = {'User-Agent': 'ResearchBot/3.2 (Academic)'}
    base_api = "https://en.wikipedia.org/w/api.php"

    try:
        # א. חיפוש כותרת
        search_params = {"action": "query", "list": "search", "format": "json", "srsearch": f"{movie_title} {movie_year} film", "srlimit": 1}
        search_resp = requests.get(base_api, params=search_params, headers=web_headers, timeout=5).json()
        if not search_resp.get('query', {}).get('search'): return entry_data
        target_page = search_resp['query']['search'][0]['title']

        # ב. שליפת תוכן
        content_params = {"action": "query", "titles": target_page, "prop": "revisions|extracts", "rvprop": "content", "explaintext": True, "format": "json", "redirects": 1}
        content_resp = requests.get(base_api, params=content_params, headers=web_headers, timeout=5).json()
        pages_dict = content_resp.get("query", {}).get("pages", {})
        pid = next(iter(pages_dict))
        if pid == "-1": return entry_data
        
        raw_wiki = pages_dict[pid].get("revisions", [{}])[0].get("*", "")
        clean_extract = pages_dict[pid].get("extract", "")

        # ג. חילוץ נתונים (Infobox)
        rules = {
            "Language": r'\|\s*language\s*=\s*([^|\n}]+)',
            "Country": r'\|\s*countr(?:y|ies)\s*=\s*([^|\n}]+)',
            "Budget": r'\|\s*budget\s*=\s*([^|\n}]+)',
            "BoxOffice": r'\|\s*(?:gross|box office)\s*=\s*([^|\n}]+)'
        }
        for key, pattern in rules.items():
            found = re.search(pattern, raw_wiki, re.IGNORECASE)
            if found:
                val = found.group(1).strip()
                val = re.sub(r'\{\{[^}]+\}\}', '', val) # ניקוי תבניות
                val = re.sub(r'\[\[(?:[^|\]]*\|)?([^\]]+)\]\]', r'\1', val) # ניקוי לינקים
                entry_data[key] = val

        # ד. חילוץ עלילה משופר
        plot_regex = re.search(r'(?:Plot|Synopsis|Summary|Premise|Story)\n+(.+)', clean_extract)
        if plot_regex:
            entry_data["Plot"] = plot_regex.group(1).strip()[:450] + "..."
        else:
            entry_data["Plot"] = clean_extract.split('\n')[0].strip()[:450] + "..."

    except: pass
    return entry_data

In [60]:
df_ultimate_combined = df_wiki_final.copy()

# 2. זיהוי השורות שעדיין חסר בהן Plot (מתוך df_wiki_final)
mask_missing = df_ultimate_combined['Plot'].isna() | (df_ultimate_combined['Plot'] == 'None') | (df_ultimate_combined['Plot'] == '')
to_fix_df = df_ultimate_combined[mask_missing]

print(f"🔄 מתחיל תיקון עבור {len(to_fix_df)} סרטים חסרים...")

start_time = time.time()
for count, (idx, row) in enumerate(to_fix_df.iterrows(), 1):

    fresh_data = fetch_movie_metadata_v3(row['primaryTitle'], row['startYear'])
    

    for col, val in fresh_data.items():
        if val:
            df_ultimate_combined.at[idx, col] = val
            
    if count % 20 == 0:
        elapsed = (time.time() - start_time) / 60
        print(f"✅ הושלמו {count}/{len(to_fix_df)} סרטים... ({elapsed:.1f} דקות)")
        # גיבוי תוך כדי תנועה לקובץ נפרד
        df_ultimate_combined.to_csv("ULTIMATE_PROGRESS_BACKUP.csv", index=False)
    
    time.sleep(1.2)

print("\n✨ תהליך הסתיים!")
print(f"הטבלה המעודכנת נמצאת ב-df_ultimate_combined")

🔄 מתחיל תיקון עבור 1743 סרטים חסרים...
✅ הושלמו 20/1743 סרטים... (0.5 דקות)
✅ הושלמו 40/1743 סרטים... (1.1 דקות)
✅ הושלמו 60/1743 סרטים... (1.7 דקות)
✅ הושלמו 80/1743 סרטים... (2.2 דקות)
✅ הושלמו 100/1743 סרטים... (2.8 דקות)
✅ הושלמו 120/1743 סרטים... (3.3 דקות)
✅ הושלמו 140/1743 סרטים... (3.8 דקות)
✅ הושלמו 160/1743 סרטים... (4.3 דקות)
✅ הושלמו 180/1743 סרטים... (4.9 דקות)
✅ הושלמו 200/1743 סרטים... (5.4 דקות)
✅ הושלמו 220/1743 סרטים... (6.0 דקות)
✅ הושלמו 240/1743 סרטים... (6.6 דקות)
✅ הושלמו 260/1743 סרטים... (7.1 דקות)
✅ הושלמו 280/1743 סרטים... (7.7 דקות)
✅ הושלמו 300/1743 סרטים... (8.2 דקות)
✅ הושלמו 320/1743 סרטים... (8.8 דקות)
✅ הושלמו 340/1743 סרטים... (9.3 דקות)
✅ הושלמו 360/1743 סרטים... (9.9 דקות)
✅ הושלמו 380/1743 סרטים... (10.4 דקות)
✅ הושלמו 400/1743 סרטים... (11.0 דקות)
✅ הושלמו 420/1743 סרטים... (11.5 דקות)
✅ הושלמו 440/1743 סרטים... (12.0 דקות)
✅ הושלמו 460/1743 סרטים... (12.6 דקות)
✅ הושלמו 480/1743 סרטים... (13.1 דקות)
✅ הושלמו 500/1743 סרטים... (13.7 דקות)
✅ הושלמו

In [61]:
df_ultimate_combined

,tconst,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,lead_actors_ids,Language,Country,Budget,BoxOffice,Plot
1423,tt0076759,Star Wars: Episode IV - A New Hope,1977.0,121.0,"Action,Adventure,Fantasy",8.6,1566475,"[nm0000434, nm0000148, nm0000402, nm0000027, n...",English,United States,$11 million,$775.4 million,"In a period of galactic civil war, Rebel Allia..."
1508,tt0080684,Star Wars: Episode V - The Empire Strikes Back,1980.0,124.0,"Action,Adventure,Fantasy",8.7,1501332,"[nm0000434, nm0000148, nm0000402, nm0001850, n...",English,United States,$30.5 million,$549–$550 million [ i ],Three years after the destruction of the Death...
1611,tt0086190,Star Wars: Episode VI - Return of the Jedi,1983.0,131.0,"Action,Adventure,Fantasy",8.3,1206896,"[nm0000434, nm0000148, nm0000402, nm0001850, n...",English,United States,$32.5–42.7 million,$482 million,One year after Han Solo's capture and imprison...
6989,tt2488496,Star Wars: Episode VII - The Force Awakens,2015.0,138.0,"Action,Adventure,Sci-Fi",7.7,1024047,"[nm5397459, nm3915784, nm1209966, nm1727304, n...",English,United States,$638.9 million (gross) $535.5 million (net),$2.071 billion,"Thirty years after the Battle of Endor,[a] the..."
4881,tt10872600,Spider-Man: No Way Home,2021.0,148.0,"Action,Adventure,Fantasy",8.1,1015964,"[nm4043618, nm4043618, nm3918035, nm1212722, n...",English,United States,$200 million,$1.921 billion,After Quentin Beck frames Peter Parker for his...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5046,tt11663918,Sweethurt,2020.0,93.0,Comedy,6.6,96,"[nm11306490, nm11306491, nm9958894, nm11306492...",None,None,None,None,None
7151,tt27164431,Space Frank,2024.0,98.0,"Comedy,Fantasy,Sci-Fi",4.6,96,"[nm2539875, nm2144988, nm0486619, nm0694907, n...",English,United Kingdom United States,$10.5 million,$148.5 million,"In a prehistoric veld, a tribe of hominins is ..."
1276,tt0070307,"Story of Karate, Fists, and Beans",1973.0,92.0,"Action,Adventure,Comedy",4.5,96,"[nm0715381, nm0400084, nm0948973, nm0562841, n...",None,None,None,None,None
5921,tt15663006,Soyrik,2022.0,109.0,\N,8.7,96,"[nm13227784, nm9717490, nm6746230, nm13227783,...",Marathi,India,None,None,The film was released on 11 February 2022 in t...


<div dir="rtl">
    
### בדיקת ניסיון שלישי

</div>

In [63]:
total_checked = len(df_ultimate_combined) # כמות סרטים בטבלה

def count_missing(column_name):
    return df_ultimate_combined[column_name].apply(lambda x: x is None or x == 'None' or str(x) == 'nan').sum() # בדיקה לכמות הערכים הטובים בטבלה 

plot_missing = count_missing('Plot')
budget_missing = count_missing('Budget')
lang_missing = count_missing('Language')
country_missing = count_missing('Country')
box_office_missing = count_missing('BoxOffice')

print(f"--- סיכום איכות נתונים (מתוך {total_checked} סרטים) ---")
print(f"עלילה חסרה:    {plot_missing} ({((plot_missing/total_checked)*100):.1f}%)")
print(f"תקציב חסר:    {budget_missing} ({((budget_missing/total_checked)*100):.1f}%)")
print(f"שפה חסרה:     {lang_missing} ({((lang_missing/total_checked)*100):.1f}%)")
print(f"מדינה חסרה:    {country_missing} ({((country_missing/total_checked)*100):.1f}%)")
print(f"הכנסות חסרות:  {box_office_missing} ({((box_office_missing/total_checked)*100):.1f}%)")

--- סיכום איכות נתונים (מתוך 5000 סרטים) ---
עלילה חסרה:    1526 (30.5%)
תקציב חסר:    3540 (70.8%)
שפה חסרה:     1205 (24.1%)
מדינה חסרה:    1312 (26.2%)
הכנסות חסרות:  3293 (65.9%)


In [64]:
# תוצאות הניסוי הראשון
# --- סיכום איכות נתונים (מתוך 5000 סרטים) ---
# עלילה חסרה:    1743 (34.9%)
# תקציב חסר:    3570 (71.4%)
# שפה חסרה:     1331 (26.6%)
# מדינה חסרה:    1337 (26.7%)
# הכנסות חסרות:  3324 (66.5%)

<div dir="rtl">
    
### ניתן לראות שחל שיפור בנתונים

</div>

In [66]:
# df_wiki_final, df_rescued_results, df_ultimate_combined שמות הטבלאות אותן מחברים

________

<div dir="rtl">
    
## החיבור
-----
</div>

In [69]:
df_master_final = df_wiki_final.copy()

# 2. הגדרת רשימת העמודות שאנחנו רוצים "לתקן"
target_cols = ['Language', 'Country', 'Budget', 'BoxOffice', 'Plot']

# פונקציית עזר לבדיקה האם ערך הוא "ריק" באמת
def is_effectively_empty(val):
    if pd.isna(val) or val is None:
        return True
    s = str(val).strip().lower()
    return s in ['nan', 'none', '', 'null', 'n/a']

# 3. מעבר על העמודות הרלוונטיות וביצוע ההשלמות
for col in target_cols:
    # לולאה על כל השורות בטבלה
    for idx in df_master_final.index:
        current_val = df_master_final.at[idx, col]
        
        # שלב א: אם הערך בבסיס ריק - נסה לקחת מ-df_rescued_results
        if is_effectively_empty(current_val):
            if idx in df_rescued_results.index:
                val_rescue = df_rescued_results.at[idx, col]
                if not is_effectively_empty(val_rescue):
                    df_master_final.at[idx, col] = val_rescue
                    current_val = val_rescue # עדכון לצורך הבדיקה הבאה
        
        # שלב ב: אם עדיין ריק - נסה לקחת מ-df_ultimate_combined
        if is_effectively_empty(current_val):
            if idx in df_ultimate_combined.index:
                val_ultimate = df_ultimate_combined.at[idx, col]
                if not is_effectively_empty(val_ultimate):
                    df_master_final.at[idx, col] = val_ultimate

print("✨ תהליך האיחוד המדורג הסתיים!")
print(f"התוצאה שמורה בכתובת: df_master_final")

# בדיקה קצרה של המצב החדש
for col in target_cols:
    missing = df_master_final[col].apply(is_effectively_empty).sum()
    print(f"עמודת {col}: נותרו {missing} ערכים חסרים.")

✨ תהליך האיחוד המדורג הסתיים!
התוצאה שמורה בכתובת: df_master_final
עמודת Language: נותרו 1200 ערכים חסרים.
עמודת Country: נותרו 1306 ערכים חסרים.
עמודת Budget: נותרו 3535 ערכים חסרים.
עמודת BoxOffice: נותרו 3289 ערכים חסרים.
עמודת Plot: נותרו 1525 ערכים חסרים.


In [70]:
df_master_final

,tconst,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,lead_actors_ids,Language,Country,Budget,BoxOffice,Plot
1423,tt0076759,Star Wars: Episode IV - A New Hope,1977.0,121.0,"Action,Adventure,Fantasy",8.6,1566475,"[nm0000434, nm0000148, nm0000402, nm0000027, n...",English,United States,$11 million,$775.4 million,"In a period of galactic civil war, Rebel Allia..."
1508,tt0080684,Star Wars: Episode V - The Empire Strikes Back,1980.0,124.0,"Action,Adventure,Fantasy",8.7,1501332,"[nm0000434, nm0000148, nm0000402, nm0001850, n...",English,United States,$30.5 million,$549–$550 million [ i ],Three years after the destruction of the Death...
1611,tt0086190,Star Wars: Episode VI - Return of the Jedi,1983.0,131.0,"Action,Adventure,Fantasy",8.3,1206896,"[nm0000434, nm0000148, nm0000402, nm0001850, n...",English,United States,$32.5–42.7 million,$482 million,One year after Han Solo's capture and imprison...
6989,tt2488496,Star Wars: Episode VII - The Force Awakens,2015.0,138.0,"Action,Adventure,Sci-Fi",7.7,1024047,"[nm5397459, nm3915784, nm1209966, nm1727304, n...",English,United States,$638.9 million (gross) $535.5 million (net),$2.071 billion,"Thirty years after the Battle of Endor,[a] the..."
4881,tt10872600,Spider-Man: No Way Home,2021.0,148.0,"Action,Adventure,Fantasy",8.1,1015964,"[nm4043618, nm4043618, nm3918035, nm1212722, n...",English,United States,$200 million,$1.921 billion,After Quentin Beck frames Peter Parker for his...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5046,tt11663918,Sweethurt,2020.0,93.0,Comedy,6.6,96,"[nm11306490, nm11306491, nm9958894, nm11306492...",None,None,None,None,None
7151,tt27164431,Space Frank,2024.0,98.0,"Comedy,Fantasy,Sci-Fi",4.6,96,"[nm2539875, nm2144988, nm0486619, nm0694907, n...",English,United Kingdom United States,$10.5 million,$148.5 million,"In a prehistoric veld, a tribe of hominins is ..."
1276,tt0070307,"Story of Karate, Fists, and Beans",1973.0,92.0,"Action,Adventure,Comedy",4.5,96,"[nm0715381, nm0400084, nm0948973, nm0562841, n...",None,None,None,None,None
5921,tt15663006,Soyrik,2022.0,109.0,\N,8.7,96,"[nm13227784, nm9717490, nm6746230, nm13227783,...",Marathi,India,None,None,The film was released on 11 February 2022 in t...


<div dir="rtl">
    
### בדיקת הנתונים הסופיים:

</div>

In [72]:
total_checked = len(df_master_final) # כמות סרטים בטבלה

def count_missing(column_name):
    return df_master_final[column_name].apply(lambda x: x is None or x == 'None' or str(x) == 'nan').sum() # בדיקה לכמות הערכים הטובים בטבלה 

plot_missing = count_missing('Plot')
budget_missing = count_missing('Budget')
lang_missing = count_missing('Language')
country_missing = count_missing('Country')
box_office_missing = count_missing('BoxOffice')

print(f"--- סיכום איכות נתונים (מתוך {total_checked} סרטים) ---")
print(f"עלילה חסרה:    {plot_missing} ({((plot_missing/total_checked)*100):.1f}%)")
print(f"תקציב חסר:    {budget_missing} ({((budget_missing/total_checked)*100):.1f}%)")
print(f"שפה חסרה:     {lang_missing} ({((lang_missing/total_checked)*100):.1f}%)")
print(f"מדינה חסרה:    {country_missing} ({((country_missing/total_checked)*100):.1f}%)")
print(f"הכנסות חסרות:  {box_office_missing} ({((box_office_missing/total_checked)*100):.1f}%)")

--- סיכום איכות נתונים (מתוך 5000 סרטים) ---
עלילה חסרה:    1525 (30.5%)
תקציב חסר:    3535 (70.7%)
שפה חסרה:     1200 (24.0%)
מדינה חסרה:    1306 (26.1%)
הכנסות חסרות:  3289 (65.8%)


In [73]:
# ניסיון ראשון
# --- סיכום איכות נתונים (מתוך 5000 סרטים) ---
# עלילה חסרה:    1743 (34.9%)
# תקציב חסר:    3570 (71.4%)
# שפה חסרה:     1331 (26.6%)
# מדינה חסרה:    1337 (26.7%)
# הכנסות חסרות:  3324 (66.5%)

In [74]:
# ניסיון שלישי
# --- סיכום איכות נתונים (מתוך 5000 סרטים) ---
# עלילה חסרה:    1526 (30.5%)
# תקציב חסר:    3540 (70.8%)
# שפה חסרה:     1205 (24.1%)
# מדינה חסרה:    1312 (26.2%)
# הכנסות חסרות:  3293 (65.9%)

<div dir="rtl">
    
### ניכר כי יש שיפור קל מהניסיון השלישי אך לא מאוד משמעותי

</div>

________

<div dir="rtl">
   
## ניקוי סופי של הטבלה
------
</div>

<div dir="rtl">
    
### בדיקה האם הפונקצייה עובדת:

</div>

In [79]:
def clean_money(val):
    if pd.isna(val) or val is None:
        return None
    
    val = str(val)
    
    # הסרת wiki templates ו-HTML comments
    val = re.sub(r'\{\{[^}]+\}\}', '', val)
    val = re.sub(r'<!--.*?-->', '', val, flags=re.DOTALL)
    val = re.sub(r'\[\s*\d+\s*\]', '', val)  # הערות כמו [i]
    val = re.sub(r'<[^>]+>', '', val)          # HTML tags
    val = val.replace('\xa0', ' ')
    
    # חיפוש כל המספרים עם יחידות
    matches = re.findall(
        r'\$?\s*([\d,]+(?:\.\d+)?)\s*(million|billion)?',
        val, re.IGNORECASE
    )
    
    values = []
    for num, unit in matches:
        num = num.replace(',', '').strip()
        if not num:
            continue
        v = float(num)
        unit = unit.lower() if unit else ''
        if unit == 'billion':
            v *= 1000
        elif unit != 'million' and v > 1000:
            v /= 1_000_000
        if 0.01 < v < 100_000:  # סינון ערכים לא הגיוניים
            values.append(v)
    
    if not values:
        return None
    return round(sum(values) / len(values), 2)

# בדיקה על הדוגמאות שלך
test_vals = [
    "550 million [ i ]",
    "535.5 million (net)",
    "{{Plainlist$10.5 million",
    "$148.5\xa0million<!-- See Box office section...",
]
for v in test_vals:
    print(f"{v[:40]:<40} → {clean_money(v)}")

550 million [ i ]                        → 550.0
535.5 million (net)                      → 535.5
{{Plainlist$10.5 million                 → 10.5
$148.5 million<!-- See Box office sectio → 148.5


<div dir="rtl">
    
### השינוי עבד ולכן נפעיל את הפונקציה על הכל

</div>

In [81]:
df_master_final["Budget"] = df_master_final["Budget"].apply(clean_money)
df_master_final["BoxOffice"] = df_master_final["BoxOffice"].apply(clean_money)

print(df_master_final[["primaryTitle", "Budget", "BoxOffice"]].dropna(subset=["Budget"]).head(10).to_string())

                                        primaryTitle  Budget  BoxOffice
1423              Star Wars: Episode IV - A New Hope    11.0      775.4
1508  Star Wars: Episode V - The Empire Strikes Back    30.5      549.5
1611      Star Wars: Episode VI - Return of the Jedi    37.6      482.0
6989      Star Wars: Episode VII - The Force Awakens   587.2     2071.0
4881                         Spider-Man: No Way Home   200.0     1921.0
3071                                          Snatch    10.0       83.6
2531                                      Spider-Man   139.0      826.8
3271                                   Spirited Away    19.2      396.0
2302       Star Wars: Episode I - The Phantom Menace   113.8     1047.0
2311    Star Wars: Episode II - Attack of the Clones   115.0      653.8


In [82]:
df_master_final

,tconst,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,lead_actors_ids,Language,Country,Budget,BoxOffice,Plot
1423,tt0076759,Star Wars: Episode IV - A New Hope,1977.0,121.0,"Action,Adventure,Fantasy",8.6,1566475,"[nm0000434, nm0000148, nm0000402, nm0000027, n...",English,United States,11.0,775.4,"In a period of galactic civil war, Rebel Allia..."
1508,tt0080684,Star Wars: Episode V - The Empire Strikes Back,1980.0,124.0,"Action,Adventure,Fantasy",8.7,1501332,"[nm0000434, nm0000148, nm0000402, nm0001850, n...",English,United States,30.5,549.5,Three years after the destruction of the Death...
1611,tt0086190,Star Wars: Episode VI - Return of the Jedi,1983.0,131.0,"Action,Adventure,Fantasy",8.3,1206896,"[nm0000434, nm0000148, nm0000402, nm0001850, n...",English,United States,37.6,482.0,One year after Han Solo's capture and imprison...
6989,tt2488496,Star Wars: Episode VII - The Force Awakens,2015.0,138.0,"Action,Adventure,Sci-Fi",7.7,1024047,"[nm5397459, nm3915784, nm1209966, nm1727304, n...",English,United States,587.2,2071.0,"Thirty years after the Battle of Endor,[a] the..."
4881,tt10872600,Spider-Man: No Way Home,2021.0,148.0,"Action,Adventure,Fantasy",8.1,1015964,"[nm4043618, nm4043618, nm3918035, nm1212722, n...",English,United States,200.0,1921.0,After Quentin Beck frames Peter Parker for his...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5046,tt11663918,Sweethurt,2020.0,93.0,Comedy,6.6,96,"[nm11306490, nm11306491, nm9958894, nm11306492...",None,None,NaN,NaN,None
7151,tt27164431,Space Frank,2024.0,98.0,"Comedy,Fantasy,Sci-Fi",4.6,96,"[nm2539875, nm2144988, nm0486619, nm0694907, n...",English,United Kingdom United States,10.5,148.5,"In a prehistoric veld, a tribe of hominins is ..."
1276,tt0070307,"Story of Karate, Fists, and Beans",1973.0,92.0,"Action,Adventure,Comedy",4.5,96,"[nm0715381, nm0400084, nm0948973, nm0562841, n...",None,None,NaN,NaN,None
5921,tt15663006,Soyrik,2022.0,109.0,\N,8.7,96,"[nm13227784, nm9717490, nm6746230, nm13227783,...",Marathi,India,NaN,NaN,The film was released on 11 February 2022 in t...


<div dir="rtl">
    
### שמירת מצב נתון:

</div>

In [84]:
df_master_final.to_csv("movies_dataset_final.csv", index=False)
print("נשמר בהצלחה!")

נשמר בהצלחה!


_______

<div dir="rtl">
    
## סידור טבלה וטיפול במשתנים:
-------
</div>

In [87]:
df_final = df_master_final[[
    "tconst", "primaryTitle", "startYear", "genres", 
    "lead_actors_ids", "runtimeMinutes", "averageRating",
    "Language", "Country", "numVotes", "Budget", "BoxOffice", "Plot"
]].copy()

df_final = df_final.rename(columns={
    "Budget": "budget",
    "BoxOffice": "BoxOffice",
    "Plot": "plot",
})
# המרת הטיפוסים בעמודות
df_final["startYear"]      = pd.to_numeric(df_final["startYear"],      errors="coerce").astype("Int64")
df_final["runtimeMinutes"] = pd.to_numeric(df_final["runtimeMinutes"], errors="coerce").astype("Int64")
df_final["numVotes"]       = pd.to_numeric(df_final["numVotes"],       errors="coerce").astype("Int64")
df_final["averageRating"]  = pd.to_numeric(df_final["averageRating"],  errors="coerce").astype(float)
df_final["budget"]         = pd.to_numeric(df_final["budget"],         errors="coerce").astype(float)
df_final["BoxOffice"]      = pd.to_numeric(df_final["BoxOffice"],      errors="coerce").astype(float)

print(df_final.dtypes)
print(f"\nצורת הטבלה: {df_final.shape}")
df_final.head(3)

tconst              object
primaryTitle        object
startYear            Int64
genres              object
lead_actors_ids     object
runtimeMinutes       Int64
averageRating      float64
Language            object
Country             object
numVotes             Int64
budget             float64
BoxOffice          float64
plot                object
dtype: object

צורת הטבלה: (5000, 13)


,tconst,primaryTitle,startYear,genres,lead_actors_ids,runtimeMinutes,averageRating,Language,Country,numVotes,budget,BoxOffice,plot
1423,tt0076759,Star Wars: Episode IV - A New Hope,1977,"Action,Adventure,Fantasy","[nm0000434, nm0000148, nm0000402, nm0000027, n...",121,8.6,English,United States,1566475,11.0,775.4,"In a period of galactic civil war, Rebel Allia..."
1508,tt0080684,Star Wars: Episode V - The Empire Strikes Back,1980,"Action,Adventure,Fantasy","[nm0000434, nm0000148, nm0000402, nm0001850, n...",124,8.7,English,United States,1501332,30.5,549.5,Three years after the destruction of the Death...
1611,tt0086190,Star Wars: Episode VI - Return of the Jedi,1983,"Action,Adventure,Fantasy","[nm0000434, nm0000148, nm0000402, nm0001850, n...",131,8.3,English,United States,1206896,37.6,482.0,One year after Han Solo's capture and imprison...


In [88]:
df_final = df_final.reset_index(drop=True) # איפוס עמודות
df_final.head(3)

,tconst,primaryTitle,startYear,genres,lead_actors_ids,runtimeMinutes,averageRating,Language,Country,numVotes,budget,BoxOffice,plot
0,tt0076759,Star Wars: Episode IV - A New Hope,1977,"Action,Adventure,Fantasy","[nm0000434, nm0000148, nm0000402, nm0000027, n...",121,8.6,English,United States,1566475,11.0,775.4,"In a period of galactic civil war, Rebel Allia..."
1,tt0080684,Star Wars: Episode V - The Empire Strikes Back,1980,"Action,Adventure,Fantasy","[nm0000434, nm0000148, nm0000402, nm0001850, n...",124,8.7,English,United States,1501332,30.5,549.5,Three years after the destruction of the Death...
2,tt0086190,Star Wars: Episode VI - Return of the Jedi,1983,"Action,Adventure,Fantasy","[nm0000434, nm0000148, nm0000402, nm0001850, n...",131,8.3,English,United States,1206896,37.6,482.0,One year after Han Solo's capture and imprison...


_________

<div dir="rtl">
    
# שמירה סופית:
-------
</div>

In [91]:
df_final.to_csv("movies_SmSz_final.csv", index=False)
print("נשמר בהצלחה!")

נשמר בהצלחה!
